In [1]:
import boto3
from datetime import datetime, timezone
import json

In [68]:
# Assume into OrganizationAccountAccessRole in a target account
# By default this uses your *current* account ID; change TARGET_ACCOUNT_ID
# if you want to assume into a different AWS account.
base_session = boto3.session.Session()
sts = base_session.client("sts")
current_identity = sts.get_caller_identity()

TARGET_ACCOUNT_ID = "060651130682"
ROLE_NAME = "OrganizationAccountAccessRole"
ROLE_ARN = f"arn:aws:iam::{TARGET_ACCOUNT_ID}:role/{ROLE_NAME}"

print("Assuming role:", ROLE_ARN)
assumed = sts.assume_role(
    RoleArn=ROLE_ARN,
    RoleSessionName="rgasa-purge-session",
)
creds = assumed["Credentials"]

# Create a session with the assumed-role credentials
assumed_session = boto3.session.Session(
    aws_access_key_id=creds["AccessKeyId"],
    aws_secret_access_key=creds["SecretAccessKey"],
    aws_session_token=creds["SessionToken"],
    region_name=base_session.region_name or "eu-west-1",
)

# Optionally make the assumed role the default for future boto3.client/resource calls
boto3.setup_default_session(
    aws_access_key_id=creds["AccessKeyId"],
    aws_secret_access_key=creds["SecretAccessKey"],
    aws_session_token=creds["SessionToken"],
    region_name=base_session.region_name or "eu-west-1",
)

# Confirm who we are now
identity = assumed_session.client("sts").get_caller_identity()

print("AWS STS get_caller_identity() after assume-role:")
print(f"  Account:   {identity['Account']}")
print(f"  UserId:    {identity['UserId']}")
print(f"  ARN:       {identity['Arn']}")
print(f"  Region:    {assumed_session.region_name}")

print("You can now use boto3.client(...) and it will use the assumed role.")

Assuming role: arn:aws:iam::060651130682:role/OrganizationAccountAccessRole
AWS STS get_caller_identity() after assume-role:
  Account:   060651130682
  UserId:    AROAIDXYY7C222M3FZUD4:rgasa-purge-session
  ARN:       arn:aws:sts::060651130682:assumed-role/OrganizationAccountAccessRole/rgasa-purge-session
  Region:    eu-west-1
You can now use boto3.client(...) and it will use the assumed role.


In [6]:
LOG_GROUP_NAME = "API-Gateway-Execution-Logs_5k58o40osi/prod"
start_day = 5
end_day = 9

start_dt = datetime(2025, 12, start_day, 0, 0, 0, tzinfo=timezone.utc)
end_dt = datetime(2025, 12, end_day, 23, 59, 59, tzinfo=timezone.utc)

output_path = f"comodash_api_logs_2025-12-{start_day}_to_2025-12-{end_day}.jsonl"


In [ ]:
# #GET the alb logs for entries with that specific apikey

# start_ms = int(start_dt.timestamp() * 1000)
# end_ms = int(end_dt.timestamp() * 1000)

# output_path = f"comodash_api_logs_2025-12-{start_day}_to_2025-12-{end_day}.jsonl"

# logs_client = boto3.client("logs")

# all_events = []
# next_token = None
# page = 0
# max_events = 10000
# filter_pattern = '1bTDa7 authorized because method "/data-input-file"'

# while True:
#     params = {
#         "logGroupName": LOG_GROUP_NAME,
#         "startTime": start_ms,
#         "endTime": end_ms,
#         "limit": 10000,
#         "filterPattern": filter_pattern,  
#     }
#     if next_token is not None:
#         params["nextToken"] = next_token

#     response = logs_client.filter_log_events(**params)
#     page += 1

#     events = response.get("events", [])
#     all_events.extend(events)

#     print(
#         f"Fetched page {page}, {len(events)} events "
#         f"(total matches so far: {len(all_events)})"
#     )

#     # if len(all_events) >= max_events:
#     #     all_events = all_events[:max_events]
#     #     break

#     next_token = response.get("nextToken")
#     if not next_token:
#         break

# with open(output_path, "w", encoding="utf-8") as f:
#     for event in all_events:
#         f.write(json.dumps(event, ensure_ascii=False))
#         f.write("\n")

# print(
#     f"Downloaded {len(all_events)} events containing '{apikey}' from '{LOG_GROUP_NAME}' "
#     f"between {start_dt} and {end_dt} into {output_path}."
# )

Fetched page 1, 0 events (total matches so far: 0)
Fetched page 2, 0 events (total matches so far: 0)
Fetched page 3, 0 events (total matches so far: 0)
Fetched page 4, 0 events (total matches so far: 0)
Fetched page 5, 0 events (total matches so far: 0)
Fetched page 6, 0 events (total matches so far: 0)
Fetched page 7, 0 events (total matches so far: 0)
Fetched page 8, 0 events (total matches so far: 0)
Fetched page 9, 0 events (total matches so far: 0)
Fetched page 10, 0 events (total matches so far: 0)
Fetched page 11, 0 events (total matches so far: 0)
Fetched page 12, 0 events (total matches so far: 0)
Fetched page 13, 0 events (total matches so far: 0)
Fetched page 14, 0 events (total matches so far: 0)
Fetched page 15, 0 events (total matches so far: 0)
Fetched page 16, 0 events (total matches so far: 0)
Fetched page 17, 0 events (total matches so far: 0)
Fetched page 18, 0 events (total matches so far: 0)
Fetched page 19, 0 events (total matches so far: 0)
Fetched page 20, 0 ev

In [8]:
import boto3

session = boto3.Session(profile_name="comodash.rgasa")
sts = session.client("sts")
print("STS from comodash.rgasa:", sts.get_caller_identity())

s3 = session.client("s3")  # <- use this s3 below

BUCKET_NAME = "comotion-comodash-rgasa-datainput"
MAX_ENTRIES = 100  # or whatever
seen_keys = set()
bucket_keys_path = "bucketKeysForDelete.txt"

with open(bucket_keys_path, "w", encoding="utf-8") as out_f, \
     open(output_path, "r", encoding="utf-8") as f:
    processed = 0
    for line in f:
        if not line.strip():
            continue
        event = json.loads(line)
        processed += 1
        # if processed > MAX_ENTRIES:
        #     break

        ts_ms = event["timestamp"]
        START_MS = ts_ms
        END_MS = ts_ms + 30_000

        START = datetime.fromtimestamp(START_MS / 1000, tz=timezone.utc)
        END = datetime.fromtimestamp(END_MS / 1000, tz=timezone.utc)

        print("START:", START, "END:", END)

        PREFIXES = ["inforce/", "terminations/", "treaty_info/"]
        matched_objects = []

        # NOTE: this block is indented INSIDE the for-line loop
        for prefix in PREFIXES:
            paginator = s3.get_paginator("list_objects_v2")
            for page in paginator.paginate(Bucket=BUCKET_NAME, Prefix=prefix):
                for obj in page.get("Contents", []):
                    last_modified = obj["LastModified"]
                    key = obj["Key"]

                    if not (START <= last_modified <= END):
                        continue

                    if key in seen_keys:
                        continue
                    seen_keys.add(key)

                    matched_objects.append({
                        "Key": key,
                        "LastModified": last_modified,
                        "Size": obj["Size"],
                    })

        print(f"Found {len(matched_objects)} new objects in {BUCKET_NAME} between {START} and {END} (UTC)")
        for o in matched_objects:
            line = f"{o['LastModified'].isoformat()}\t{o['Size']}\t{o['Key']}"
            print(line)
            out_f.write(line + "\n")
        print("########################################################")

STS from comodash.rgasa: {'UserId': 'AROAJ5YTGMWHKRNBYDR4Y:botocore-session-1768307865', 'Account': '396511695522', 'Arn': 'arn:aws:sts::396511695522:assumed-role/OrganizationAccountAccessRole/botocore-session-1768307865', 'ResponseMetadata': {'RequestId': 'aebbf7c5-a75c-4222-982f-9933ecb500f3', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': 'aebbf7c5-a75c-4222-982f-9933ecb500f3', 'x-amz-sts-extended-request-id': 'MTpldS13ZXN0LTE6UzoxNzY4MzA3ODY3NTM2OlI6T3h5MmR6bno=', 'content-type': 'text/xml', 'content-length': '490', 'date': 'Tue, 13 Jan 2026 12:37:47 GMT'}, 'RetryAttempts': 0}}
START: 2025-12-08 07:52:07.713000+00:00 END: 2025-12-08 07:52:37.713000+00:00
Found 6 new objects in comotion-comodash-rgasa-datainput between 2025-12-08 07:52:07.713000+00:00 and 2025-12-08 07:52:37.713000+00:00 (UTC)
2025-12-08T07:52:10+00:00	1659632	inforce/data_import_batch=2025-12-08/service_client_id=0/cc01b1f0-d40a-11f0-b095-3379fb785f2b.csv.gz
2025-12-08T07:52:14+00:00	1783131	inforce/dat

In [ ]:
# #Delete all the objects in the s3 bucket metioned in the log file
# import boto3

# BUCKET_NAME = "comotion-comodash-rgasa-datainput"
# PATH = "bucketKeysForDelete.txt"
# BATCH_SIZE = 1000  # max 1000 keys per delete_objects call

# # Use existing assumed-role session if you have one, otherwise this creates a default client
# s3 = boto3.client("s3")

# # 1) Read keys from the file (3rd column in each tab-separated line)
# keys = []
# with open(PATH, "r", encoding="utf-8") as f:
#     for line in f:
#         line = line.strip()
#         if not line:
#             continue
#         parts = line.split("\t")
#         if len(parts) < 3:
#             continue
#         key = parts[2]
#         keys.append(key)

# # Optional: deduplicate keys
# unique_keys = sorted(set(keys))
# print(f"Read {len(keys)} keys, {len(unique_keys)} unique keys")

# # 2) Delete in batches
# for i in range(0, len(unique_keys), BATCH_SIZE):
#     batch = unique_keys[i:i + BATCH_SIZE]
#     delete_payload = {"Objects": [{"Key": k} for k in batch], "Quiet": False}
#     response = s3.delete_objects(Bucket=BUCKET_NAME, Delete=delete_payload)

#     deleted = response.get("Deleted", [])
#     errors = response.get("Errors", [])
#     print(f"Batch {i // BATCH_SIZE + 1}: requested={len(batch)}, deleted={len(deleted)}, errors={len(errors)}")

#     if errors:
#         print("Some errors occurred:")
#         for e in errors:
#             print(f"  Key={e.get('Key')} Code={e.get('Code')} Message={e.get('Message')}")